# 02 Generate the AI reviews (Colab, open models)

For a sample of human reviews we generate one AI review each, matched on **game, thumbs up/down and length**, so that
topic and length cannot give the class away. Two open models from Hugging Face run on Colab's free GPU:

| Generator | Model | Role |
| --- | --- | --- |
| `mistral` | `mistralai/Mistral-7B-Instruct-v0.3` | training + test |
| `qwen` | `Qwen/Qwen2.5-7B-Instruct` | **held out**: never used in training, only as a test of an unseen LLM |

Decisions (each one is defendable, see `failure-journal.md`, 18 Nov 2026):

- **One prompt = one review.** Asking for 20 reviews in one answer makes them influence each other (repeated openings, drifting style).
  Batching on the GPU gives the speed instead.
- **The prompt contains only the game name, the recommendation, a target length and a style.** Never the human review text,
  so we do not get paraphrases of the human reviews.
- **Five prompt styles**, picked at random per review, so the detector cannot learn one prompt's voice.
- **Human reviews over 300 words are not paired** (2.6% of the cleaned data): 7B models do not keep long target lengths
  and long generations are slow. This is a stated limit.
- The AI reviews go through **the same cleaning rules** as the human ones (`scripts/clean_reviews.py`), plus removal of
  chat giveaways ("Here is a review:", quotation marks, markdown).

**How to run**

1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `reviews_clean.csv` (v3) to Google Drive: `MyDrive/TA2/data/clean/v3/reviews_clean.csv`.
3. Mistral is a "gated" model: log in on huggingface.co, open the model page and accept the conditions. Then create a
   token (Settings → Access Tokens, type *Read*) and add it in Colab under 🔑 Secrets as `HF_TOKEN`.
4. Set `GENERATOR` below, then Runtime → Run all. First run with `TEST_RUN = True` (100 reviews) to see the speed.
5. When Colab disconnects, just run all again: finished reviews are on Drive and are skipped.
6. After `mistral` is finished, set `GENERATOR = "qwen"`, **restart the runtime** (frees the GPU) and run all again.
7. When both are done, section 5 (assembly) puts everything together. It works without a GPU.
8. Then run notebook **02b_fix_lengths** (reviews whose length is far from the human twin's are regenerated).

## 1. Settings

In [ ]:
GENERATOR = "mistral"   # "mistral" or "qwen"
TEST_RUN = True         # True: only 100 reviews, to measure speed. Set False for the real run.

N_PER_GAME = 1000       # human reviews sampled per game -> 12,000 pairs
HELD_OUT_SHARE = 0.25   # share of pairs generated by the held-out model (qwen) -> 3,000
MAX_WORDS = 300         # longer human reviews are not paired
BATCH_SIZE = 16         # prompts generated at once on the GPU; lower it if you get "CUDA out of memory"
USE_LENGTH_FIX = False  # notebook 02: False (original lengths). Notebook 02b sets it to True.
RUN_LENGTH_FIX = USE_LENGTH_FIX
FIX_CANDIDATES = 4      # (02b) versions generated per review in the length fix; the closest to the target is kept
SEED = 42

MODELS = {
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen": "Qwen/Qwen2.5-7B-Instruct",
}
HELD_OUT_GENERATOR = "qwen"

# Sampling settings, the same for both models (recorded with every review)
GEN_SETTINGS = dict(do_sample=True, temperature=0.9, top_p=0.95, repetition_penalty=1.05)

DRIVE_DIR = "/content/drive/MyDrive/TA2"
HUMAN_CSV = f"{DRIVE_DIR}/data/clean/v3/reviews_clean.csv"
OUT_DIR = f"{DRIVE_DIR}/data/ai/v1"

## 2. Setup: Google Drive, the repository (for the cleaning rules), libraries

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

# The repository is public: we clone it to reuse exactly the same cleaning rules as for the human reviews.
if not os.path.exists("/content/ta2-semestral-project"):
    !git clone -q https://github.com/nike03156/ta2-semestral-project /content/ta2-semestral-project
sys.path.append("/content/ta2-semestral-project/scripts")

!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import json, random, re, time
from pathlib import Path
import pandas as pd
import clean_reviews as cr   # our own cleaning script from the repository

os.makedirs(OUT_DIR, exist_ok=True)

## 3. Pairs: which human reviews get an AI twin, from which model, in which style

The pairs are drawn once (seed 42) and saved to Drive as `pairs.csv`, so both generator runs use the same table and
every AI review can be traced back to its human review (`pair_id` = the human `review_id`).

In [ ]:
GAMES = {
    620: "Portal 2", 105600: "Terraria", 292030: "The Witcher 3: Wild Hunt", 413150: "Stardew Valley",
    289070: "Sid Meier's Civilization VI", 367520: "Hollow Knight", 578080: "PUBG: Battlegrounds",
    646570: "Slay the Spire", 1145360: "Hades", 1151340: "Fallout 76", 1091500: "Cyberpunk 2077",
    1517290: "Battlefield 2042",
}

# The style is the only thing that changes the "voice" of the prompt. Length and recommendation come from the pair.
STYLES = {
    "casual":    "Write it casually, like typing it quickly.",
    "detailed":  "Mention one or two concrete things about the gameplay.",
    "emotional": "Let your feelings show clearly.",
    "humorous":  "Use a bit of humor.",
    "plain":     "Keep it plain and to the point.",
}

pairs_path = Path(OUT_DIR) / "pairs.csv"
if pairs_path.exists():
    pairs = pd.read_csv(pairs_path)
    print("Loaded existing pairs.csv")
else:
    human = pd.read_csv(HUMAN_CSV)
    eligible = human[human.word_count <= MAX_WORDS]
    print(f"Human reviews: {len(human)}; with at most {MAX_WORDS} words: {len(eligible)} "
          f"({1 - len(eligible) / len(human):.1%} excluded)")
    assert eligible.groupby("appid").size().min() >= N_PER_GAME, "a game has fewer reviews than N_PER_GAME"
    sample = eligible.groupby("appid").sample(n=N_PER_GAME, random_state=SEED)
    rng = random.Random(SEED)
    pairs = pd.DataFrame({
        "pair_id": sample.review_id.values,
        "appid": sample.appid.values,
        "game": sample.appid.map(GAMES).values,
        "voted_up": sample.voted_up.values,
        "target_words": sample.word_count.values,
    })
    pairs["prompt_style"] = [rng.choice(sorted(STYLES)) for _ in range(len(pairs))]
    # Randomly give HELD_OUT_SHARE of the pairs to the held-out model, the rest to the other one.
    order = list(range(len(pairs))); rng.shuffle(order)
    n_held = round(HELD_OUT_SHARE * len(pairs))
    pairs["generator"] = "mistral"
    pairs.loc[order[:n_held], "generator"] = HELD_OUT_GENERATOR
    pairs.to_csv(pairs_path, index=False)
    print("Saved", pairs_path)

print(pairs.groupby("generator").size())
pairs.head()

In [ ]:
def build_prompt(game, voted_up, target_words, style, length_rule="about"):
    """The whole instruction the model sees. Only game, recommendation, length and style; no human text.

    length_rule: "about" (normal run), or "at most" / "at least" (length fix, section 4b).
    """
    verdict = "recommend" if voted_up else "do not recommend"
    # Prompt v2 (18 Nov): v1 ("Write a user review of...") produced advert-like text, see failure journal.
    return (f'You have played the video game "{game}" and you are writing your own user review of it on Steam.\n'
            f"You {verdict} the game.\n"
            f"Length: {length_rule} {target_words} words.\n"
            f"{STYLES[style]}\n"
            "Write it the way real players write Steam reviews: first person, your own opinion, "
            "not like an advertisement or a magazine article.\n"
            "Reply with the review text only: no title, no rating, no quotation marks, no introduction.")

print(build_prompt("Hades", True, 19, "casual"))

Post-processing: strips what a chat model adds around the review (used in sections 4b and 5).

In [ ]:
PREAMBLE = re.compile(r"^(sure|certainly|of course|here('s| is)|okay|ok)\b[^\n]*:\s*$", re.I)
LABEL = re.compile(r"^(\*\*)?(steam )?(user )?review( title)?(\*\*)?:\s*", re.I)
FOOTER = re.compile(r"^\(?(word count|words?)\s*:?\s*\d+\)?\.?$|^-{3,}$", re.I)
MARKDOWN = re.compile(r"\*\*|__|^#+\s*", re.M)
QUOTES = "\"'“”‘’"

def postprocess(raw):
    """Strip chat wrappers. Returns (text, list of what was removed)."""
    removed = []
    lines = [l.strip() for l in raw.strip().splitlines()]
    while lines and (not lines[0] or PREAMBLE.match(lines[0])):
        if lines[0]: removed.append("preamble")
        lines.pop(0)
    while lines and (not lines[-1] or FOOTER.match(lines[-1])):
        if lines[-1]: removed.append("footer")
        lines.pop()
    text = "\n".join(lines)
    if LABEL.match(text):
        text = LABEL.sub("", text, count=1); removed.append("label")
    if MARKDOWN.search(text):
        text = MARKDOWN.sub("", text); removed.append("markdown")
    if len(text) > 1 and text[0] in QUOTES and text[-1] in QUOTES:
        text = text[1:-1].strip(); removed.append("quotes")
    return text, removed

## 4. Generation (needs the GPU)

In [ ]:
import torch
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

login(userdata.get("HF_TOKEN"))
model_id = MODELS[GENERATOR]

# 4-bit loading: a 7B model needs ~14.5 GB in fp16, which does not fit next to the batch on a 15 GB T4.
# In 4 bits it takes ~5 GB. Quality loss for short texts like reviews is small.
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"   # decoder models must be padded on the left for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant, device_map="auto")
model.eval()
print("Loaded", model_id)

In [ ]:
def generate_batch(prompts, max_new_tokens):
    """Generate one answer per prompt. Returns the raw model outputs (only the new text)."""
    chats = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                           add_generation_prompt=True) for p in prompts]
    inputs = tokenizer(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                             **GEN_SETTINGS)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

The loop writes every finished batch to `raw_<generator>.jsonl` on Drive immediately. On a restart it reads that file
and skips the pairs that are already done. Pairs are sorted by target length, so a batch holds reviews of similar
length and the GPU does not wait on one long review.

In [ ]:
raw_path = Path(OUT_DIR) / f"raw_{GENERATOR}.jsonl"
done = set()
if raw_path.exists():
    with raw_path.open(encoding="utf-8") as f:
        done = {json.loads(line)["pair_id"] for line in f if line.strip()}

todo = pairs[(pairs.generator == GENERATOR) & ~pairs.pair_id.isin(done)].sort_values("target_words")
if TEST_RUN:
    todo = todo.sample(min(100, len(todo)), random_state=SEED).sort_values("target_words")
print(f"{GENERATOR}: {len(done)} done, {len(todo)} to generate now")

start, n_written = time.time(), 0
with raw_path.open("a", encoding="utf-8") as f:
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[i:i + BATCH_SIZE]
        prompts = [build_prompt(r.game, r.voted_up, r.target_words, r.prompt_style) for r in batch.itertuples()]
        # ~1.3 tokens per English word; generous room, the prompt asks for the length anyway
        max_new = int(batch.target_words.max() * 2) + 40
        torch.manual_seed(SEED + i)
        outputs = generate_batch(prompts, max_new)
        for r, prompt, text in zip(batch.itertuples(), prompts, outputs):
            f.write(json.dumps({"pair_id": r.pair_id, "generator": GENERATOR, "model": MODELS[GENERATOR],
                                "prompt_style": r.prompt_style, "target_words": int(r.target_words),
                                "prompt": prompt, "raw_text": text, "settings": GEN_SETTINGS},
                               ensure_ascii=False) + "\n")
        f.flush()
        n_written += len(batch)
        if (i // BATCH_SIZE) % 10 == 0 or n_written == len(todo):   # progress every 10 batches
            rate = n_written / (time.time() - start)
            print(f"{n_written}/{len(todo)}  {rate * 60:.0f} reviews/min  "
                  f"about {(len(todo) - n_written) / rate / 60:.0f} min left")
print(f"Finished {n_written} reviews in {(time.time() - start) / 60:.1f} min")

## 5. Assembly: post-processing and the same cleaning as the human reviews

Run this when both generators are done (no GPU needed). It reads `raw_mistral.jsonl` and `raw_qwen.jsonl`.

**Post-processing** removes what a chat model adds around the review and a human reviewer would never write.
Every change is counted and examples are saved, so we can show in the report that it removes only these wrappers.

In [ ]:
from collections import Counter
from statistics import median

rows = []
for gen in MODELS:
    path = Path(OUT_DIR) / f"raw_{gen}.jsonl"
    if not path.exists():
        continue
    with path.open(encoding="utf-8") as f:
        by_pair = {r["pair_id"]: r for r in map(json.loads, filter(str.strip, f))}
    fix = Path(OUT_DIR) / f"raw_{gen}_lengthfix.jsonl"
    if USE_LENGTH_FIX and fix.exists():   # notebook 02b: a closer-length version replaces the original
        with fix.open(encoding="utf-8") as f:
            for r in map(json.loads, filter(str.strip, f)):
                if r["length_fix"]:
                    by_pair[r["pair_id"]] = r
    rows += list(by_pair.values())
print("Raw AI reviews:", len(rows), dict(Counter(r["generator"] for r in rows)))
print("Replaced by the length fix:", dict(Counter(r["generator"] for r in rows if r.get("length_fix"))))

meta = pairs.set_index("pair_id")
report, post_counts, kept, examples, seen = Counter(), Counter(), [], [], set()
for r in rows:
    report["input"] += 1
    text, removed = postprocess(r["raw_text"])
    post_counts.update(removed)
    if removed and len(examples) < 40:
        examples.append({"generator": r["generator"], "removed": "+".join(removed),
                         "raw_text": r["raw_text"], "text": text})
    # the same rules, in the same order, as for the human reviews (language tag: none for AI text)
    text = cr.WHITESPACE.sub(" ", text).strip()
    check = cr.normalize(text)
    words = cr.WORD.findall(check)
    reason = cr.drop_reason({"language": None}, check, words, 3, seen)
    if reason:
        report[reason] += 1
        continue
    seen.add(cr.duplicate_key(check))
    report["kept"] += 1
    m = meta.loc[r["pair_id"]]
    kept.append({"review_id": f"ai_{r['generator']}_{r['pair_id']}", "pair_id": r["pair_id"],
                 "appid": int(m.appid), "text": text, "voted_up": bool(m.voted_up), "word_count": len(words),
                 "target_words": r["target_words"], "generator": r["generator"],
                 "prompt_style": r["prompt_style"], "held_out": r["generator"] == HELD_OUT_GENERATOR,
                 "length_fix": bool(r.get("length_fix", False)),
                 "label": 1})

ai = pd.DataFrame(kept)
ai.to_csv(Path(OUT_DIR) / "ai_reviews_clean.csv", index=False)
pd.DataFrame(examples).to_csv(Path(OUT_DIR) / "postprocess_examples.csv", index=False)

summary = {"cleaning": dict(report), "postprocess_removed": dict(post_counts),
           "kept_per_generator": ai.groupby("generator").size().to_dict(),
           "length_fixed_per_generator": ai[ai.length_fix].groupby("generator").size().to_dict(),
           "median_words_ai": float(ai.word_count.median()),
           "median_words_target": float(ai.target_words.median()),
           "median_abs_length_error": float((ai.word_count - ai.target_words).abs().median())}
(Path(OUT_DIR) / "generation_report.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

### Quick checks before using the data

- **Length match**: does the AI length follow the target? If one generator is always much longer, length alone could
  separate the classes, and the pairing did not work.
- Read `postprocess_examples.csv` and a random sample of AI reviews per generator and style. Note anything
  obviously artificial in the failure journal (with an example).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
for gen, g in ai.groupby("generator"):
    ax.scatter(g.target_words, g.word_count, s=4, alpha=0.4, label=gen)
ax.plot([0, MAX_WORDS], [0, MAX_WORDS], "k--", lw=1)
ax.set(xlabel="target words (human twin)", ylabel="AI words", title="Length match")
ax.legend(); plt.show()

for gen, g in ai.groupby("generator"):
    print(f"--- {gen} ---")
    for t in g.sample(min(5, len(g)), random_state=SEED).text:
        print("*", t[:300])